## PMT-935

In [0]:
env="p"

In [0]:
spark.sql("""
MERGE INTO `p-cdh`.landed.extfile_report_unified_ml_vw AS a
USING `p-cdh`.landed.extfile_report_unified_ml AS b
ON a.businessdate = b.BusinessDate 
   AND a.account_number = b.account_number 
   AND a.loan_application_id = b.loan_application_id
    AND a.step_description = b.step_description
WHEN MATCHED AND a.DateEvalReceived IS NULL THEN
    UPDATE SET a.DateEvalReceived = b.date_eval_received;
 """)

In [0]:
spark.sql("""
MERGE INTO `p-cdh`.landed.extfile_report_unified_ml_vw AS a
USING `p-cdh`.landed.extfile_report_unified_ml AS b
ON a.businessdate = b.BusinessDate 
   AND a.account_number = b.account_number 
   AND a.loan_application_id = b.loan_application_id 
   AND a.step_description = b.step_description
WHEN MATCHED AND a.status IS NULL THEN
    UPDATE SET a.status = b.status;
""")

## addition of lookup_export table

In [0]:
%sql
drop table `p-cdh`.reference.lookup_export

In [0]:
%sql
create table `p-cdh`.reference.lookup_export
as
select * from `d-cdh`.reference.lookup_export

##DWH-3500

In [0]:
spark.sql("""
update `p-cdh`.primaryactive.portfolio
set    TiresiasCode='TOP'   ,
       TiresiasName='TOP INVESTMENTS AE'
where PortfolioName='TOP INVESTMENTS AE'
""")

## DD-46

In [0]:
spark.sql(f"""
ALTER TABLE `p-cdh`.derivativeactive.dexposure
ADD COLUMN TotalUnrealizedInterest decimal(19,2) AFTER DueDebtAmount;
""")

In [0]:
spark.sql(f"""
ALTER TABLE `p-cdh`.derivativearchive.dexposure
ADD COLUMN TotalUnrealizedInterest decimal(19,2) AFTER DueDebtAmount;
""")

In [0]:
spark.sql(f"""
MERGE INTO `p-cdh`.derivativeactive.dexposure AS target
USING (
    SELECT ExposureDerivativeID, BusinessDate, Unrealizedinterest, DueUnrealizedinterest
    FROM `p-cdh`.derivativeactive.dexposure
) AS source
ON target.ExposureDerivativeID = source.ExposureDerivativeID
AND target.BusinessDate = source.BusinessDate
WHEN MATCHED THEN
UPDATE SET target.TotalUnrealizedInterest = coalesce(source.Unrealizedinterest, lit(0)) + coalesce(source.DueUnrealizedinterest, lit(0));
""")

In [0]:

spark.sql(f"""
ALTER TABLE `p-cdh`.derivativeactive.dexposure
ADD COLUMN NetDueAmount decimal(19,2) AFTER TotalUnrealizedInterest;
""")

In [0]:
spark.sql(f"""
ALTER TABLE `p-cdh`.derivativearchive.dexposure
ADD COLUMN NetDueAmount decimal(19,2) AFTER TotalUnrealizedInterest;
""")

In [0]:
spark.sql(f"""
MERGE INTO `p-cdh`.derivativeactive.dexposure AS target
USING (
    SELECT ExposureDerivativeID, BusinessDate, DuePrincipal, Dueinterest, DueUnrealizedinterest, DueExpense
    FROM `p-cdh`.derivativeactive.dexposure
) AS source
ON target.ExposureDerivativeID = source.ExposureDerivativeID
AND target.BusinessDate = source.BusinessDate
WHEN MATCHED THEN
UPDATE SET target.NetDueAmount = coalesce(DuePrincipal, lit(0)) + coalesce(Dueinterest, lit(0)) 
                               + coalesce(DueUnrealizedinterest, lit(0)) + coalesce(DueExpense, lit(0));
""")

In [0]:
spark.sql(f"""
ALTER TABLE `p-cdh`.derivativeactive.dexposure
ADD COLUMN TotalInterestAccruals decimal(19,2) AFTER NetDueAmount;
""")


In [0]:
spark.sql(f"""
ALTER TABLE `p-cdh`.derivativearchive.dexposure
ADD COLUMN TotalInterestAccruals decimal(19,2) AFTER NetDueAmount;
""")

In [0]:
spark.sql(f"""
MERGE INTO `p-cdh`.derivativeactive.dexposure AS target
USING (
    SELECT ExposureDerivativeID, BusinessDate, interest, Dueinterest, DueUnrealizedinterest, Penaltyinterest
        , PenaltyUnrealizedinterest, Unrealizedinterest, Accrued, DueAccrued
    FROM `p-cdh`.derivativeactive.dexposure
) AS source
ON target.ExposureDerivativeID = source.ExposureDerivativeID
AND target.BusinessDate = source.BusinessDate
WHEN MATCHED THEN
UPDATE SET target.TotalInterestAccruals = coalesce(interest, lit(0)) + coalesce(Dueinterest, lit(0)) + 
                                          coalesce(DueUnrealizedinterest, lit(0)) + coalesce(Penaltyinterest, lit(0)) + 
                                          coalesce(PenaltyUnrealizedinterest, lit(0)) + coalesce(Unrealizedinterest, lit(0)) + 
                                          coalesce(Accrued, lit(0)) + coalesce(DueAccrued, lit(0));
""")

In [0]:
spark.sql(f"""
ALTER TABLE `p-cdh`.derivativeactive.dexposure
ADD COLUMN TotalInterestL3869 decimal(19,2) AFTER TotalInterestAccruals;
""")

In [0]:
spark.sql(f"""
ALTER TABLE `p-cdh`.derivativearchive.dexposure
ADD COLUMN TotalInterestL3869 decimal(19,2) AFTER TotalInterestAccruals;
""")

In [0]:
spark.sql(f"""
MERGE INTO `p-cdh`.derivativeactive.dexposure AS target
USING (
    SELECT ExposureDerivativeID, BusinessDate, TotalInterest
    FROM `p-cdh`.derivativeactive.dexposure
    WHERE ExposureStatus = 'ΕΠΙΣΦΑΛΕΙΑ'
    AND SourceSystem = 'profits'
    AND (ProductKey like '%4015' or
         ProductKey like '%4016' or
         ProductKey like '%6078' or
         ProductKey like '%6079' or
         ProductKey like '%7362' or
         ProductKey like '%7363' or
         ProductKey like '%40000' or
         ProductKey like '%400001')
) AS source
ON target.ExposureDerivativeID = source.ExposureDerivativeID
AND target.BusinessDate = source.BusinessDate
WHEN MATCHED THEN
UPDATE SET target.TotalInterestL3869 = TotalInterest
""")

## Refresh views

In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)



In [0]:
# Get the schema of the dexposure table
dexposure_table = spark.table(f"`{env}-cdh`.derivative.dexposure")
dexposure_schema_fields = dexposure_table.schema.fields


# Convert the schema fields into the required string format
dexposure_schema = ",\n".join([
    f"{field.name} {field.dataType.simpleString()}"
    for field in dexposure_schema_fields
])


spark.sql(f'''
CREATE OR REPLACE FUNCTION `{env}-cdh`.derivative.get_all_exposures_by_business_date(
    business_date Date
)
RETURNS TABLE 
( 
{dexposure_schema}
)
RETURN

SELECT B.*
FROM(
        SELECT  ExposureKey, MAX(BusinessDate) BusinessDate
        FROM `{env}-cdh`.derivative.dexposure
        WHERE BusinessDate <= business_date
        GROUP BY ExposureKey
) A
INNER JOIN   `{env}-cdh`.derivative.dexposure B
on a.ExposureKey = b.ExposureKey
and a.BusinessDate = b.BusinessDate
AND B.IsActiveFlagInt = 0

UNION

SELECT * FROM `{env}-cdh`.derivative.dexposure
WHERE BusinessDate = business_date
AND IsActiveFlagInt = 1
''')